# Tahap Data Acquisition 


# Library Yang Dibutuhkan

In [1]:
# Import library dasar untuk manipulasi file dan data
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # untuk menghindari error pada beberapa sistem dan memakai dua dua versi library OpenMP 
import shutil #untuk mengatur file dan folder (seperti menghapus, memindah, menyalin, dll)
from pathlib import Path

# Import library untuk numerik dan visualisasi
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import library untuk image processing
from PIL import Image
import cv2

# Import library PyTorch (untuk cek apakah environment sudah siap)
import torch
import torchvision

# Konfigurasi visual
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print(f" PyTorch Version: {torch.__version__}")
print(f" Torchvision Version: {torchvision.__version__}")
print(f" CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Device: {torch.cuda.get_device_name(0)}")


 PyTorch Version: 2.7.1+cu118
 Torchvision Version: 0.22.1+cu118
 CUDA Available: True
   GPU Device: NVIDIA GeForce RTX 3060


# Pembagian data train, val, test dengan rasio 79:10:10


In [2]:
import os
import shutil
import random
from collections import defaultdict, Counter

def parse_group_id(filename):
    """
    Ekstrak ID subjek dari awal nama file MRL.
    Contoh: "s0001_00001_0_0_0_0_0_01.png" -> "s0001"
    """
    base = os.path.splitext(os.path.basename(filename))[0]
    parts = base.split("_")
    return parts[0] if len(parts) > 0 else base

def split_dataset_mrl_final(source_dir, dest_dir):
    # ===========================
    # PEMBAGIAN ID SUBJEK HASIL AUDIT
    # ===========================
    # Kelompok ini dipilih agar proporsi JUMLAH GAMBAR mendekati 80:10:10
    VAL_IDS = ['s0008', 's0021', 's0006', 's0004', 's0002', 's0015', 's0026', 's0030']
    TEST_IDS = ['s0022', 's0009', 's0010', 's0023', 's0007', 's0035', 's0027', 's0003', 's0033', 's0020', 's0005', 's0028', 's0034', 's0024']
    
    # 1. Cek ketersediaan kelas
    classes = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
    classes = sorted(classes) # Biasanya ["Close-Eyes", "Open-Eyes"]
    
    if not classes:
        print(" [!] Tidak menemukan folder kelas. Pastikan path SOURCE benar.")
        return

    # 2. Buat struktur folder tujuan
    splits_list = ["train", "val", "test"]
    for split in splits_list:
        for cls in classes:
            os.makedirs(os.path.join(dest_dir, split, cls), exist_ok=True)

    print(f"Mulai memproses dataset dari: {source_dir}")
    print(f"Hasil akan disimpan di   : {dest_dir}\n")

    # 3. Iterasi file dan copy berdasarkan ID
    counts = Counter()

    for cls in classes:
        cls_dir = os.path.join(source_dir, cls)
        files = [f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg",".jpeg",".png"))]
        
        print(f"Memproses kelas {cls} ({len(files)} gambar)...")
        
        for f in files:
            gid = parse_group_id(f)
            
            # Tentukan split berdasarkan ID subjek
            if gid in TEST_IDS:
                target_split = "test"
            elif gid in VAL_IDS:
                target_split = "val"
            else:
                target_split = "train"
            
            src_path = os.path.join(cls_dir, f)
            dst_path = os.path.join(dest_dir, target_split, cls, f)
            
            shutil.copy2(src_path, dst_path)
            counts[target_split] += 1

    # 4. Laporan Hasil
    total_all = sum(counts.values())
    print("\n" + "="*50)
    print(f"{'LAPORAN AKHIR SPLIT DATASET MRL':^50}")
    print("="*50)
    for s in splits_list:
        perc = (counts[s]/total_all)*100 if total_all > 0 else 0
        print(f" {s.upper():5} SET : {counts[s]:6} gambar ({perc:.2f}%)")
    print("-" * 50)
    print(f" TOTAL DATA : {total_all} gambar")
    print("="*50)

def verify_mrl_leakage(dest_dir):
    splits = ["train", "val", "test"]
    split_groups = {}

    for s in splits:
        ids = set()
        split_path = os.path.join(dest_dir, s)
        for root, dirs, files in os.walk(split_path):
            for f in files:
                if f.lower().endswith((".jpg", ".png", ".jpeg")):
                    ids.add(parse_group_id(f))
        split_groups[s] = ids

    print("\n=== VERIFIKASI SUBJEK (ANTI LEAKAGE) ===")
    print(f"Subjek di Train : {len(split_groups['train'])} orang")
    print(f"Subjek di Val   : {len(split_groups['val'])} orang")
    print(f"Subjek di Test  : {len(split_groups['test'])} orang")

    # Cek irisan (Intersection)
    leak_tv = split_groups["train"] & split_groups["val"]
    leak_tt = split_groups["train"] & split_groups["test"]
    
    if not leak_tv and not leak_tt:
        print("\n[STATUS] AMAN 100%! Tidak ada subjek yang bocor antar folder.")
    else:
        print("\n[WARNING] Terjadi kebocoran subjek!")

# =====================================================================
# EKSEKUSI
# =====================================================================
# Sesuaikan path ini dengan folder di laptopmu
source_dataset = r"C:\kuliah-sementara\SKRIPSI\Dataset_MRL" 
dest_dataset = r"C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT"

if os.path.exists(dest_dataset):
    print(f"Menghapus folder lama: {dest_dataset}")
    shutil.rmtree(dest_dataset)

split_dataset_mrl_final(source_dataset, dest_dataset)
verify_mrl_leakage(dest_dataset)

Mulai memproses dataset dari: C:\kuliah-sementara\SKRIPSI\Dataset_MRL
Hasil akan disimpan di   : C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT

Memproses kelas Close-Eyes (41946 gambar)...
Memproses kelas Open-Eyes (42952 gambar)...

         LAPORAN AKHIR SPLIT DATASET MRL          
 TRAIN SET :  67474 gambar (79.48%)
 VAL   SET :   8776 gambar (10.34%)
 TEST  SET :   8648 gambar (10.19%)
--------------------------------------------------
 TOTAL DATA : 84898 gambar

=== VERIFIKASI SUBJEK (ANTI LEAKAGE) ===
Subjek di Train : 15 orang
Subjek di Val   : 8 orang
Subjek di Test  : 14 orang

[STATUS] AMAN 100%! Tidak ada subjek yang bocor antar folder.


# Konfigurasi Path Dataset

In [12]:
# ===========================
# UBAH PATH INI SESUAI LOKASI DATASET ANDA
# ===========================

# Path utama dataset (parent folder) yang sudah di-split
DATASET_ROOT = r"C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT" 

# Path untuk setiap split
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "val")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

# Nama kelas (folder di dalam train/val/test)
CLASS_NAMES = ["Open-Eyes", "Close-Eyes"]  

# Cek apakah folder-folder ini ada
print(" Memeriksa struktur folder dataset...\n")

for split_name, split_path in [("Train", TRAIN_DIR), ("Val", VAL_DIR), ("Test", TEST_DIR)]:
    if os.path.exists(split_path):
        print(f" {split_name} folder ditemukan: {split_path}")
        for class_name in CLASS_NAMES:
            class_path = os.path.join(split_path, class_name)
            if os.path.exists(class_path):
                print(f"   ├─  Kelas '{class_name}' ditemukan")
            else:
                print(f"   ├─  Kelas '{class_name}' TIDAK DITEMUKAN!")
    else:
        print(f" {split_name} folder TIDAK DITEMUKAN: {split_path}")

print("\n" + "="*60)


 Memeriksa struktur folder dataset...

 Train folder ditemukan: C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT\train
   ├─  Kelas 'Open-Eyes' ditemukan
   ├─  Kelas 'Close-Eyes' ditemukan
 Val folder ditemukan: C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT\val
   ├─  Kelas 'Open-Eyes' ditemukan
   ├─  Kelas 'Close-Eyes' ditemukan
 Test folder ditemukan: C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT\test
   ├─  Kelas 'Open-Eyes' ditemukan
   ├─  Kelas 'Close-Eyes' ditemukan



# Fungsi untuk Menghitung Jumlah File per Kelas

In [13]:
def count_images_in_folder(folder_path):
    """
    Menghitung jumlah file gambar (.jpg, .png, .jpeg) di dalam folder.
    
    Args:
        folder_path (str): Path ke folder yang akan dihitung.
        
    Returns:
        int: Jumlah file gambar.
    """
    if not os.path.exists(folder_path):
        return 0
    
    valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    count = 0
    
    for file in os.listdir(folder_path):
        if any(file.lower().endswith(ext) for ext in valid_extensions):
            count += 1
    
    return count


def get_dataset_statistics(base_dir, class_names):
    """
    Menghitung statistik dataset untuk setiap split dan kelas.
    
    Args:
        base_dir (str): Path ke folder split (train/val/test).
        class_names (list): Daftar nama kelas (misalnya ['wake', 'sleep']).
        
    Returns:
        dict: Dictionary berisi jumlah gambar per kelas.
    """
    stats = {}
    
    for class_name in class_names:
        class_path = os.path.join(base_dir, class_name)
        stats[class_name] = count_images_in_folder(class_path)
    
    return stats

# Menghitung dan Menampilkan Statistik Dataset

In [15]:
# Hitung jumlah gambar di setiap split
print(" STATISTIK DATASET\n")
print("="*60)

splits = {
    "Train": TRAIN_DIR,
    "Val": VAL_DIR,
    "Test": TEST_DIR
}

all_stats = {}

for split_name, split_path in splits.items():
    stats = get_dataset_statistics(split_path, CLASS_NAMES)
    all_stats[split_name] = stats
    
    total = sum(stats.values())
    
    print(f"\n {split_name.upper()} SET:")
    print(f"   ├─ Open-Eyes (Wake): {stats['Open-Eyes']:,} gambar")
    print(f"   ├─ Close-Eyes (Sleep): {stats['Close-Eyes']:,} gambar")
    print(f"   └─ Total: {total:,} gambar")
    
    # Hitung balance ratio
    if total > 0:
        Open_ratio = (stats['Open-Eyes'] / total) * 100
        Close_ratio = (stats['Close-Eyes'] / total) * 100
        print(f"      Balance: {Open_ratio:.2f}% Open-Eyes | {Close_ratio:.2f}% Close-Eyes")

print("\n" + "="*60)

# Ringkasan total keseluruhan
total_all = sum([sum(stats.values()) for stats in all_stats.values()])
print(f"\n TOTAL DATASET: {total_all:,} gambar")


 STATISTIK DATASET


 TRAIN SET:
   ├─ Open-Eyes (Wake): 34,439 gambar
   ├─ Close-Eyes (Sleep): 33,035 gambar
   └─ Total: 67,474 gambar
      Balance: 51.04% Open-Eyes | 48.96% Close-Eyes

 VAL SET:
   ├─ Open-Eyes (Wake): 3,198 gambar
   ├─ Close-Eyes (Sleep): 5,578 gambar
   └─ Total: 8,776 gambar
      Balance: 36.44% Open-Eyes | 63.56% Close-Eyes

 TEST SET:
   ├─ Open-Eyes (Wake): 5,315 gambar
   ├─ Close-Eyes (Sleep): 3,333 gambar
   └─ Total: 8,648 gambar
      Balance: 61.46% Open-Eyes | 38.54% Close-Eyes


 TOTAL DATASET: 84,898 gambar


# Fungsi Cek dan Hapus File Corrupt

In [16]:
def check_and_remove_corrupt_images(base_dir, class_names):
    """
    Memeriksa semua gambar di dalam folder dan menghapus file yang corrupt.
    
    Args:
        base_dir (str): Path ke folder split (train/val/test).
        class_names (list): Daftar nama kelas.
        
    Returns:
        list: Daftar file yang dihapus karena corrupt.
    """
    corrupt_files = []
    
    for class_name in class_names:
        class_path = os.path.join(base_dir, class_name)
        
        if not os.path.exists(class_path):
            print(f"  Folder tidak ditemukan: {class_path}")
            continue
        
        print(f"\n Memeriksa folder: {class_path}")
        
        for filename in os.listdir(class_path):
            file_path = os.path.join(class_path, filename)
            
            # Skip jika bukan file
            if not os.path.isfile(file_path):
                continue
            
            try:
                # Coba buka gambar menggunakan PIL
                img = Image.open(file_path)
                img.verify()  # Verifikasi integritas file
                
                # Coba load sebagai array (double check)
                img = Image.open(file_path)
                img.load()
                
            except Exception as e:
                print(f"    File corrupt: {filename} - Error: {str(e)}")
                corrupt_files.append(file_path)
                
                # Hapus file corrupt
                try:
                    os.remove(file_path)
                    print(f"        File telah dihapus.")
                except Exception as del_error:
                    print(f"        Gagal menghapus: {str(del_error)}")
    
    return corrupt_files

# Jalankan Pengecekan File Corrupt

In [17]:
print(" MEMULAI PENGECEKAN FILE CORRUPT...\n")
print("="*60)

all_corrupt_files = []

for split_name, split_path in splits.items():
    print(f"\n{'='*60}")
    print(f" SPLIT: {split_name.upper()}")
    print(f"{'='*60}")
    
    corrupt_files = check_and_remove_corrupt_images(split_path, CLASS_NAMES)
    all_corrupt_files.extend(corrupt_files)

print("\n" + "="*60)
print(f"\n HASIL PENGECEKAN:")
print(f"   Total file corrupt yang ditemukan dan dihapus: {len(all_corrupt_files)}")

if len(all_corrupt_files) == 0:
    print("    SEMUA FILE DALAM KONDISI BAIK!")
else:
    print("\n   Daftar file yang dihapus:")
    for file in all_corrupt_files:
        print(f"   - {file}")

print("="*60)


 MEMULAI PENGECEKAN FILE CORRUPT...


 SPLIT: TRAIN

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT\train\Open-Eyes

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT\train\Close-Eyes

 SPLIT: VAL

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT\val\Open-Eyes

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT\val\Close-Eyes

 SPLIT: TEST

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT\test\Open-Eyes

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_MRL_SPLIT\test\Close-Eyes


 HASIL PENGECEKAN:
   Total file corrupt yang ditemukan dan dihapus: 0
    SEMUA FILE DALAM KONDISI BAIK!


# Visualisasi Sampel Gambar dari Dataset

In [19]:
def calculate_dataset_stats(base_dir, class_names):
    """
    Menghitung jumlah gambar dalam setiap kelas di sebuah direktori.
    """
    stats = {}
    for class_name in class_names:
        class_path = os.path.join(base_dir, class_name)
        if os.path.exists(class_path):
            files = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]
            stats[class_name] = len(files)
        else:
            stats[class_name] = 0
            
    return stats

print(" DISTRIBUSI KELAS DATASET\n")
print("="*60)

total_all = 0
split_stats = {}

for split_name, split_path in splits.items():
    stats = calculate_dataset_stats(split_path, CLASS_NAMES)
    split_stats[split_name] = stats
    
    total = sum(stats.values())
    total_all += total
    
    print(f"\n SPLIT: {split_name.upper()}")
    print("-" * 30)
    print(f"   ├─ Open-Eyes (Sleep): {stats.get('Open-Eyes', 0):,} gambar")
    print(f"   ├─ Close-Eyes (Wake): {stats.get('Close-Eyes', 0):,} gambar")
    print(f"   └─ Total {split_name}: {total:,} gambar")
    
    if total > 0:
        Open_ratio = (stats.get('Open-Eyes', 0) / total) * 100
        Closed_ratio = (stats.get('Close-Eyes', 0) / total) * 100
        print(f"      Balance: {Open_ratio:.2f}% Open-Eyes | {Closed_ratio:.2f}% Close-Eyes")

print("\n" + "="*60)
print(f" TOTAL KESELURUHAN DATASET: {total_all:,} gambar")
print("="*60)


 DISTRIBUSI KELAS DATASET


 SPLIT: TRAIN
------------------------------
   ├─ Open-Eyes (Sleep): 34,439 gambar
   ├─ Close-Eyes (Wake): 33,035 gambar
   └─ Total Train: 67,474 gambar
      Balance: 51.04% Open-Eyes | 48.96% Close-Eyes

 SPLIT: VAL
------------------------------
   ├─ Open-Eyes (Sleep): 3,198 gambar
   ├─ Close-Eyes (Wake): 5,578 gambar
   └─ Total Val: 8,776 gambar
      Balance: 36.44% Open-Eyes | 63.56% Close-Eyes

 SPLIT: TEST
------------------------------
   ├─ Open-Eyes (Sleep): 5,315 gambar
   ├─ Close-Eyes (Wake): 3,333 gambar
   └─ Total Test: 8,648 gambar
      Balance: 61.46% Open-Eyes | 38.54% Close-Eyes

 TOTAL KESELURUHAN DATASET: 84,898 gambar


# Menghitung Mean dan Std Pixel (untuk Normalisasi Custom)

In [20]:
def calculate_mean_std(base_dir, class_names, num_samples=5000):
    """
    Menghitung mean dan std dari pixel dataset untuk normalisasi custom.
    Dioptimalkan menggunakan float32 dan penutupan file memori secara eksplisit 
    agar tidak menyebabkan RAM bocor atau CPU freeze.
    """
    print(f"\n Menghitung Mean & Std dari {num_samples} sampel gambar (Optimized Mode)...\n")
    
    sample_count = 0
    pixel_num = 0
    # Menggunakan float64 hanya untuk akumulator akhir agar menghindari overflow
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sum_sq = np.zeros(3, dtype=np.float64)
    
    for class_name in class_names:
        class_path = os.path.join(base_dir, class_name)
        
        if not os.path.exists(class_path):
            continue
        
        all_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        
        if len(all_files) == 0:
            continue
            
        samples_per_class = min(num_samples // len(class_names), len(all_files))
        sample_files = np.random.choice(all_files, samples_per_class, replace=False)
        
        for filename in sample_files:
            img_path = os.path.join(class_path, filename)
            
            try:
                # Menggunakan context manager (with) agar file langsung ditutup dari RAM setelah dibaca
                with Image.open(img_path) as img:
                    img = img.convert('RGB')
                    # Menggunakan float32 alih-alih float64 untuk menghemat 50% RAM per proses gambar
                    img_array = np.array(img, dtype=np.float32) / 255.0  
                
                # Menghitung sum langsung tanpa reshape yang membuat copy array baru
                # axis=(0, 1) akan menjumlahkan nilai di sumbu tinggi dan lebar gambar, menyisakan 3 nilai (R, G, B)
                channel_sum += np.sum(img_array, axis=(0, 1), dtype=np.float64)
                channel_sum_sq += np.sum(img_array**2, axis=(0, 1), dtype=np.float64)
                
                # Menambah jumlah total piksel (tinggi x lebar)
                pixel_num += (img_array.shape[0] * img_array.shape[1])
                
                sample_count += 1
                
                # Feedback progres ke terminal agar tidak terlihat freeze
                if sample_count % 1000 == 0:
                    print(f"   Terproses {sample_count} gambar...")
                    
            except Exception as e:
                # Diam jika ada gambar rusak sebagian
                pass
    
    # Hitung mean dan std akhir
    mean = channel_sum / pixel_num
    std = np.sqrt((channel_sum_sq / pixel_num) - mean**2)
    
    print(f"\n Kalkulasi selesai dari {sample_count} gambar.")
    print(f"\n HASIL:")
    print(f"   Mean (R, G, B): [{mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f}]")
    print(f"   Std  (R, G, B): [{std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f}]")
    
    return mean, std


# Hitung mean & std dari Training Set. 
# Jika PC masih terasa berat, ubah num_samples ke 5000 atau 2000
mean, std = calculate_mean_std(TRAIN_DIR, CLASS_NAMES, num_samples=5000)

print("\n CATATAN:")
print("   Nilai Mean & Std ini akan digunakan untuk normalisasi di tahap preprocessing.")
print("   Simpan nilai ini untuk digunakan di notebook berikutnya!")


 Menghitung Mean & Std dari 5000 sampel gambar (Optimized Mode)...

   Terproses 1000 gambar...
   Terproses 2000 gambar...
   Terproses 3000 gambar...
   Terproses 4000 gambar...
   Terproses 5000 gambar...

 Kalkulasi selesai dari 5000 gambar.

 HASIL:
   Mean (R, G, B): [0.3772, 0.3772, 0.3772]
   Std  (R, G, B): [0.1544, 0.1544, 0.1544]

 CATATAN:
   Nilai Mean & Std ini akan digunakan untuk normalisasi di tahap preprocessing.
   Simpan nilai ini untuk digunakan di notebook berikutnya!


# Menghitung nilai mean dan std secara spesifik dari dataset target (seperti Ntuhd) merupakan standar profesional daripada menggunakan nilai bawaan ImageNet, terutama karena citra mata (grayscale/inframerah) memiliki distribusi piksel yang sangat berbeda dari foto natural warna-warni. Pendekatan ini memastikan model belajar mengenali fitur pada skala intensitas yang tepat sesuai domain aslinya. Agar ketahanan model (robustness) tetap terjaga saat implementasi ke kamera real-time (webcam), hasil crop mata berformat RGB dari kamera wajib diubah menjadi grayscale terlebih dahulu, lalu dikonversi kembali menjadi format RGB 3-channel tiruan sebelum dinormalisasi menggunakan mean dan std Ntud tersebut, sehingga input inferensi memiliki karakteristik visual yang identik sempurna dengan data saat training.

# Chek Duplikasi


In [21]:
import hashlib
import os

def calculate_md5(file_path):
    """Menghitung MD5 hash dari sebuah file gambar."""
    hash_md5 = hashlib.md5()
    try:
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def find_and_remove_duplicates(train_dir, val_dir, test_dir):
    """
    Mencari dan MENGHAPUS gambar duplikat antar folder split.
    Prioritas Hapus: Hapus file di Test/Val, pertahankan file di Train.
    """
    print(" SEDANG MEMERIKSA & MEMBERSIHKAN DUPLIKASI KONTEN...")
    print("   (Proses ini mungkin memakan waktu 1-2 menit...)")
    
    train_hashes = {}
    val_hashes = {}
    test_hashes = {}
    
    def fill_hashes(directory, hash_dict, label):
        print(f"    Memproses {label} set...")
        count = 0
        for root, _, files in os.walk(directory):
            for file in files:
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    path = os.path.join(root, file)
                    file_hash = calculate_md5(path)
                    if file_hash:
                        hash_dict[file_hash] = path
                        count += 1
        return count

    # 1. Hitung Hash
    fill_hashes(train_dir, train_hashes, "TRAIN")
    fill_hashes(val_dir, val_hashes, "VAL")
    fill_hashes(test_dir, test_hashes, "TEST")
    
    # 2. Cek Irisan & Hapus
    train_keys = set(train_hashes.keys())
    val_keys = set(val_hashes.keys())
    test_keys = set(test_hashes.keys())
    
    leak_train_test = train_keys.intersection(test_keys)
    leak_train_val = train_keys.intersection(val_keys)
    
    print("\n HASIL PEMBERSIHAN:")
    print("="*60)
    
    # Hapus Duplikat Train vs Test (Hapus yang di Test)
    if leak_train_test:
        print(f" Ditemukan {len(leak_train_test)} duplikat di TRAIN & TEST. Menghapus file di TEST...")
        for h in leak_train_test:
            file_to_remove = test_hashes[h]
            try:
                os.remove(file_to_remove)
                print(f"    Dihapus: {file_to_remove}")
            except Exception as e:
                print(f"    Gagal hapus {file_to_remove}: {e}")
    else:
        print(" Train vs Test: AMAN (Tidak ada duplikat).")

    # Hapus Duplikat Train vs Val (Hapus yang di Val)
    if leak_train_val:
        print(f" Ditemukan {len(leak_train_val)} duplikat di TRAIN & VAL. Menghapus file di VAL...")
        for h in leak_train_val:
            file_to_remove = val_hashes[h]
            try:
                os.remove(file_to_remove)
                print(f"    Dihapus: {file_to_remove}")
            except Exception as e:
                print(f"    Gagal hapus {file_to_remove}: {e}")
    else:
        print(" Train vs Val: AMAN (Tidak ada duplikat).")
        
    print("="*60)
    print(" Dataset sekarang sudah BERSIH dan VALID!")

# Jalankan Fungsi
find_and_remove_duplicates(TRAIN_DIR, VAL_DIR, TEST_DIR)


 SEDANG MEMERIKSA & MEMBERSIHKAN DUPLIKASI KONTEN...
   (Proses ini mungkin memakan waktu 1-2 menit...)
    Memproses TRAIN set...
    Memproses VAL set...
    Memproses TEST set...

 HASIL PEMBERSIHAN:
 Train vs Test: AMAN (Tidak ada duplikat).
 Train vs Val: AMAN (Tidak ada duplikat).
 Dataset sekarang sudah BERSIH dan VALID!


# Cek Data Leakage (Kebocoran Data Antar Split)

In [22]:
def check_data_leakage_by_filename(train_dir, val_dir, test_dir):
    """
    Mengecek kebocoran data berdasarkan NAMA FILE (Frame ID).
    Logika Baru:
    - Tidak masalah subjek (s0001) sama di Train/Test.
    - Yang HARAM adalah jika NAMA FILE (Frame) sama persis muncul di Train & Test.
    """
    print(" MEMERIKSA DATA LEAKAGE (BERDASARKAN DUPLIKASI NAMA FILE)...")
    print("="*60)

    # Helper untuk ambil semua nama file (tanpa path)
    def get_all_filenames(directory):
        filenames = set()
        count = 0
        for root, _, files in os.walk(directory):
            for file in files:
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    filenames.add(file)
                    count += 1
        return filenames, count

    # Ambil daftar nama file
    train_files, n_train = get_all_filenames(train_dir)
    val_files, n_val = get_all_filenames(val_dir)
    test_files, n_test = get_all_filenames(test_dir)

    print(f"   Jumlah Gambar di Train : {n_train}")
    print(f"   Jumlah Gambar di Val   : {n_val}")
    print(f"   Jumlah Gambar di Test  : {n_test}")

    # Cek Irisan (Apakah ada nama file yang sama persis?)
    leak_train_test = train_files.intersection(test_files)
    leak_train_val = train_files.intersection(val_files)
    leak_val_test = val_files.intersection(test_files)

    print("\n HASIL ANALISIS LEAKAGE:")
    
    # Laporan 1: Train vs Test
    if len(leak_train_test) > 0:
        print(f"    PERINGATAN: Ditemukan {len(leak_train_test)} file dengan nama SAMA di Train & Test!")
        print(f"      Ini berpotensi data leakage (Model diuji dengan gambar yang sama saat latihan).")
        print(f"      Contoh: {list(leak_train_test)[:3]}")
    else:
        print("    AMAN: Tidak ada file dengan nama yang sama antara Train & Test.")
        print("      (Validitas pengujian terjamin karena frame-nya unik).")

    # Laporan 2: Train vs Val
    if len(leak_train_val) > 0:
        print(f"    PERINGATAN: Ditemukan {len(leak_train_val)} file dengan nama SAMA di Train & Val!")
    else:
        print("    AMAN: Tidak ada file dengan nama yang sama antara Train & Val.")

    print("="*60)
    print(" KESIMPULAN AKHIR:")
    if len(leak_train_test) == 0 and len(leak_train_val) == 0:
        print("   Dataset VALID. Meskipun subjek (orang) sama muncul di berbagai split,")
        print("   tapi Frame/Gambar yang digunakan berbeda. Model akan belajar generalisasi pose mata,")
        print("   bukan menghafal identitas orang.")
    else:
        print("   Masih ada duplikasi nama file. Pastikan Anda sudah menjalankan kode penghapusan duplikat sebelumnya.")

# Jalankan Fungsi Revisi
check_data_leakage_by_filename(TRAIN_DIR, VAL_DIR, TEST_DIR)


 MEMERIKSA DATA LEAKAGE (BERDASARKAN DUPLIKASI NAMA FILE)...
   Jumlah Gambar di Train : 67474
   Jumlah Gambar di Val   : 8776
   Jumlah Gambar di Test  : 8648

 HASIL ANALISIS LEAKAGE:
    AMAN: Tidak ada file dengan nama yang sama antara Train & Test.
      (Validitas pengujian terjamin karena frame-nya unik).
    AMAN: Tidak ada file dengan nama yang sama antara Train & Val.
 KESIMPULAN AKHIR:
   Dataset VALID. Meskipun subjek (orang) sama muncul di berbagai split,
   tapi Frame/Gambar yang digunakan berbeda. Model akan belajar generalisasi pose mata,
   bukan menghafal identitas orang.


<!-- # print("\n" + "="*60)
# print(" NOTEBOOK 01: DATA PREPARATION & EXPLORATION - SELESAI!")
# print("="*60)
# print("\n RINGKASAN:")
# print("   ✓ Struktur folder dataset sudah dicek")
# print("   ✓ Statistik dataset sudah dihitung dan divisualisasi")
# print("   ✓ File corrupt sudah dicek dan dihapus (jika ada)")
# print("   ✓ Sampel gambar sudah ditampilkan")
# print("   ✓ Mean & Std pixel sudah dihitung untuk normalisasi")
# print("\n LANGKAH SELANJUTNYA:")
# print("   → Lanjut ke Notebook 02: Preprocessing Pipeline Check")
# print("="*60) -->